In [ ]:
# ==== CONFIG (edit these paths/params) ====
POS_PATH = "positives_TSS.csv"   # positives file
NEG_PATH = "negatives_TSS.csv"   # negatives file
OUT_PREFIX = "tss_rfe"           # output prefix (no extension)

# RFECV (auto-select #features) or RFE (fixed #features)
USE_RFECV  = False        # True -> RFECV; False -> RFE
N_FEATURES = 120         # used only when USE_RFECV == False

# Cross-validation
CV_FOLDS     = 5
USE_GROUP_CV = True      # if True and 'chr' exists in both files, use GroupKFold

# Estimator
ESTIMATOR = "logreg"     # 'logreg' or 'linearsvc'
SCORING   = "recall"  # or 'roc_auc'
RANDOM_STATE = 13


In [ ]:
import os, re, numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.feature_selection import RFE, RFECV
import matplotlib.pyplot as plt

def load_align(pos_path, neg_path):
    pos = pd.read_csv(pos_path)
    neg = pd.read_csv(neg_path)
    common = [c for c in pos.columns if c in neg.columns]
    return pos[common].copy(), neg[common].copy()

def infer_feature_columns(df, pattern=r"_TSS_bin\d+$"):
    feats = [c for c in df.columns if re.search(pattern, c)]
    if not feats:
        feats = [c for c in df.columns if re.search(r"_bin\d+$", c)]
    meta = {"gene","chr","strand","TSS_coord","TTS_coord","y","label","class"}
    feats = [c for c in feats if c not in meta]
    if not feats:
        raise ValueError("No feature columns matching *_TSS_binNN (or *_binNN) found.")
    return feats

def make_estimator(name):
    if name == "logreg":
        est = LogisticRegression(penalty="l2", solver="liblinear",
                                 class_weight="balanced", max_iter=2000,
                                 random_state=RANDOM_STATE)
    elif name == "linearsvc":
        est = LinearSVC(penalty="l2", dual=False, class_weight="balanced",
                        max_iter=5000, random_state=RANDOM_STATE)
    else:
        raise ValueError("ESTIMATOR must be 'logreg' or 'linearsvc'")
    pipe = Pipeline([("scaler", StandardScaler(with_mean=True)), ("clf", est)])
    return pipe, "named_steps.clf.coef_"

def make_cv(use_group, folds, groups):
    if use_group and (groups is not None):
        return GroupKFold(n_splits=folds)
    return StratifiedKFold(n_splits=folds, shuffle=True, random_state=RANDOM_STATE)


In [ ]:
pos_df, neg_df = load_align(POS_PATH, NEG_PATH)
feat_cols = infer_feature_columns(pos_df)
meta_cols = [c for c in ["gene","chr","strand","TSS_coord"] if c in pos_df.columns and c in neg_df.columns]

X = pd.concat([pos_df[feat_cols], neg_df[feat_cols]], axis=0).reset_index(drop=True)
y = np.r_[np.ones(len(pos_df), dtype=int), np.zeros(len(neg_df), dtype=int)]
meta = pd.concat([pos_df[meta_cols], neg_df[meta_cols]], axis=0).reset_index(drop=True) if meta_cols else None

groups = None
if USE_GROUP_CV and ("chr" in pos_df.columns and "chr" in neg_df.columns):
    groups = pd.concat([pos_df["chr"], neg_df["chr"]], axis=0).to_numpy()

print(f"Pos={len(pos_df)} Neg={len(neg_df)} | Features={len(feat_cols)} | Meta={meta_cols}")
X.shape, y.shape


In [ ]:
est, imp_getter = make_estimator(ESTIMATOR)
cv = make_cv(USE_GROUP_CV, CV_FOLDS, groups)

if USE_RFECV:
    selector = RFECV(
        estimator=est,
        step=0.1,
        min_features_to_select=max(10, int(0.05*X.shape[1])),
        cv=cv,
        scoring=SCORING,
        importance_getter=imp_getter,
        n_jobs=1
    )
    selector.fit(X.values, y, groups=groups)
    support = selector.support_
    ranking = selector.ranking_
    print(f"RFECV selected {support.sum()}/{X.shape[1]} features")
else:
    selector = RFE(
        estimator=est,
        n_features_to_select=min(int(N_FEATURES), X.shape[1]),
        step=0.1,
        importance_getter=imp_getter
    )
    selector.fit(X.values, y)
    support = selector.support_
    ranking = selector.ranking_
    print(f"RFE selected {support.sum()}/{X.shape[1]} features")

selected = [f for f, keep in zip(feat_cols, support) if keep]
print("First 10 selected:", selected[:10])
len(selected)


In [ ]:
os.makedirs(os.path.dirname(OUT_PREFIX) or ".", exist_ok=True)

# 1) Selected features
with open(f"{OUT_PREFIX}_selected_features.txt", "w") as f:
    for name in selected:
        f.write(name + "\n")

# 2) Support mask + ranking
mask_df = pd.DataFrame({"feature": feat_cols, "keep": support.astype(bool), "ranking": ranking})
mask_df.to_csv(f"{OUT_PREFIX}_support_mask.csv", index=False)

# 3) Reduced dataset (meta + y + selected features)
X_sel = X.values[:, support]
reduced = pd.DataFrame(X_sel, columns=selected)
reduced.insert(0, "y", y)
if meta is not None:
    for c in meta.columns[::-1]:
        reduced.insert(0, c, meta[c].to_numpy())
out_reduced = f"{OUT_PREFIX}_reduced_120.csv"
reduced.to_csv(out_reduced, index=False)

print("Saved:")
print(f" - {OUT_PREFIX}_selected_feature_120.txt")
print(f" - {OUT_PREFIX}_support_mask_120.csv")
print(f" - {out_reduced}")


In [ ]:
# Force-write CV scores for both RFECV and RFE (robust across sklearn versions)
import numpy as np, os
from sklearn.model_selection import cross_val_score

out_scores = f"{OUT_PREFIX}_cv_scores_120.txt"
os.makedirs(os.path.dirname(OUT_PREFIX) or ".", exist_ok=True)

def _write_pairs(ns, scores, path):
    with open(path, "w") as f:
        for n, s in zip(ns, scores):
            f.write(f"{int(n)}\t{float(s):.6f}\n")
    print("Saved:", path, f"({len(list(ns))} points)")

if USE_RFECV:
    ns, means = None, None

    # Try native RFECV outputs first
    if hasattr(selector, "cv_results_") and isinstance(selector.cv_results_, dict) \
       and "mean_test_score" in selector.cv_results_:
        means = np.asarray(selector.cv_results_["mean_test_score"])
        # Some versions expose selector.n_features_ (array); otherwise build a linear range
        if hasattr(selector, "n_features_") and np.ndim(selector.n_features_) > 0:
            ns = selector.n_features_
        else:
            start = int(getattr(selector, "min_features_to_select",
                                max(10, int(0.05*X.shape[1]))))
            ns = np.arange(start, start + len(means))

    elif hasattr(selector, "grid_scores_"):
        # Older sklearn
        means = np.asarray(selector.grid_scores_)
        start = int(getattr(selector, "min_features_to_select",
                            max(10, int(0.05*X.shape[1]))))
        ns = np.arange(start, start + len(means))

    if means is None:
        # Manual curve: evaluate a grid of k using selector.ranking_
        order = np.argsort(selector.ranking_)  # best (rank=1) first
        p = X.shape[1]
        start = max(10, int(0.05 * p))
        step = max(1, (p - start) // 25)   # ~25 points
        ns = list(range(start, p + 1, step))
        means = []
        for k in ns:
            mask = np.zeros(p, dtype=bool); mask[order[:k]] = True
            Xk = X.values[:, mask]
            sc = cross_val_score(est, Xk, y, cv=cv, scoring=SCORING, groups=groups)
            means.append(sc.mean())

    _write_pairs(ns, means, out_scores)

else:
    # RFE: score the final selected subset
    X_sel = X.values[:, support]
    sc = cross_val_score(est, X_sel, y, cv=cv, scoring=SCORING, groups=groups)
    with open(out_scores, "w") as f:
        f.write(f"mean_{SCORING}={sc.mean():.6f}\tstd={sc.std():.6f}\n")
    print("Saved:", out_scores, "| mean:", round(sc.mean(), 6), "std:", round(sc.std(), 6))


In [ ]:
TOPN = 30  # adjust
top = imp_df.head(TOPN).iloc[::-1]  # reverse for horizontal bar
plt.figure(figsize=(8, max(4, TOPN*0.25)))
plt.barh(top["feature"], top["importance"])
plt.xlabel("Absolute coefficient")
plt.ylabel("Feature")
plt.title(f"Top {TOPN} selected features by |coef|")
plt.tight_layout()
plt.show()


In [ ]:
sel_groups = group_by_mark(selected_cols)
mark_counts = pd.Series({m: len(v) for m, v in sel_groups.items()}).sort_values(ascending=False)

plt.figure(figsize=(8, max(3, 0.3*len(mark_counts))))
plt.barh(mark_counts.index, mark_counts.values)
plt.xlabel("# Selected bins")
plt.ylabel("Mark")
plt.title("Selected bins per mark")
plt.tight_layout()
plt.show()

mark_counts


In [ ]:
# Plot mean profiles across bins for positives vs negatives for chosen marks
# (Assumes features are exactly 20 bins per mark: *_TSS_bin01..20)
marks_to_plot = list(mark_counts.index[:4])  # top 4 marks; edit as needed

pos_mask = (y == 1)
neg_mask = (y == 0)

ncols = 2
nrows = int(np.ceil(len(marks_to_plot)/ncols))
plt.figure(figsize=(10, 3*nrows))
idx = 1
for mark in marks_to_plot:
    # pick columns of this mark, sorted by bin
    mark_feats = [f for f in feat_cols if f.startswith(mark+"_") and re.search(r"_bin\d+$", f)]
    mark_feats = sorted(mark_feats, key=lambda f: parse_mark_bin(f)[1] or 0)

    # take from ALL features (before selection) so you see the full 20-bin profile
    Xm = X[mark_feats].to_numpy()
    pos_mean = Xm[pos_mask].mean(axis=0)
    neg_mean = Xm[neg_mask].mean(axis=0)

    plt.subplot(nrows, ncols, idx); idx += 1
    plt.plot(range(1, len(mark_feats)+1), pos_mean, label="Pos (TSS)")
    plt.plot(range(1, len(mark_feats)+1), neg_mean, label="Neg")
    plt.title(f"{mark}: mean profile")
    plt.xlabel("Bin (1..20)")
    plt.ylabel("Mean signal")
    plt.legend()
    plt.tight_layout()

plt.show()


In [ ]:
# Heatmap for top-K features by |coef|
K = 50  # adjust
heat_cols = imp_df.head(K)["feature"].tolist()
Xm = X[heat_cols]
# z-score columns for visualization
Xm_z = (Xm - Xm.mean(axis=0)) / (Xm.std(axis=0) + 1e-9)

order = np.argsort(y)[::-1]  # positives first
plt.figure(figsize=(10, 6))
plt.imshow(Xm_z.to_numpy()[order, :], aspect="auto", interpolation="nearest")
plt.colorbar(label="z-score")
plt.yticks([])
plt.xticks(range(len(heat_cols)), heat_cols, rotation=90)
plt.title(f"Heatmap: top {K} features (|coef|), samples sorted by label")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.manifold import TSNE

X_sel_np, _ = take_cols(X, selected)
# Standardize before TSNE for stability
Xs = (X_sel_np - X_sel_np.mean(axis=0)) / (X_sel_np.std(axis=0) + 1e-9)

tsne = TSNE(n_components=2, perplexity=30, learning_rate="auto", init="pca", random_state=RANDOM_STATE)
Z = tsne.fit_transform(Xs)

plt.figure(figsize=(6,5))
plt.scatter(Z[y==1,0], Z[y==1,1], s=10, label="Pos", alpha=0.7)
plt.scatter(Z[y==0,0], Z[y==0,1], s=10, label="Neg", alpha=0.7)
plt.legend()
plt.title("t-SNE on selected features")
plt.tight_layout()
plt.show()
